# Projeto COP30 - Classificação de Projetos Sustentáveis


**Contexto:** Estamos em setembro de 2025, a poucos meses da **COP30 em Belém (novembro/2025)**. Este estudo aproveita o contexto da **COP30 em Belém (novembro/2025)**, e o desafio é identificar projetos que realmente tenham potencial de transformar compromissos climáticos em resultados concretos.

O objetivo é usar **ciência de dados** para apoiar decisões estratégicas: priorizar **projetos sustentáveis** com maior potencial de impacto ambiental e social, e boa viabilidade econômica/operacional, considerando múltiplos fatores ambientais, sociais, econômicos e tecnológicos.

**Tarefa:** Dado um *dataset* com indicadores ambientais, sociais, econômicos e tecnológicos, prever se um projeto é **Viável** ou **Não Viável** usando algoritmos de **classificação** (Árvore de Decisão, SVM e k-NN).




## Roteiro (CRISP-DM)

1. **Compreensão do Negócio:** critérios (CO2, água, circularidade, resiliência, maturirdade tecnológica, custos/retorno(.
2. **Compreensão dos Dados:** leitura, dicinário, qualidade,valores ausentes.
3. **Preparação dos Dados:** *encoding*, imputação, padronização (quando necessário).
4. **Modelagem:** Decision Tree, SVM, KNN (pipelines).
5. **Avaliação:** acurávia, precisão, recall, F1, matriz de confusão, ROC-AUC (binário).
6. **Reflexão/Deploy:** interpretação, trade-off e conexão com decisões COP30.

## Imports e verificação de versões

In [1]:
!pip install pandas numpy scikit-learn matplotlib joblib


In [2]:
import sys, platform                            # infos do Python/SO(documentação)
import numpy as np                              # cálculo numérico/vetorial
import pandas as pd                             # manipulação tabula(DataFrame)
import matplotlib.pyplot as plt                 # Gráficos

# Ferramentas de modelagem/validação do scikit-learn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Modelos
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

#Métricas
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_curve, auc
)
import joblib   # salvar/carregar modelos treinados

print(f"Python: {sys.version.split()[0]} | OS: {platform.system()} {platform.release()}\n")


Python: 3.11.11 | OS: Darwin 24.6.0



## 1.LEITURA DO DATASET — Compreensão dos Dados

**O que faz**: Lê o arquivo csv e cria um DataFrame.  
**Por que**: Precisamos inspecionar os dados antes de decidir como preparar e modelar.  


In [3]:

# Configuração
DATAPATH = "dataset_socioambiental_60.csv"

# Importação
df = pd.read_csv(DATAPATH)

# Mostra dimensões e primeiras linhas
print("Dimensão do dataset:", df.shape)
display(df.head(10))  # mostra 10 linhas

FileNotFoundError: [Errno 2] No such file or directory: 'dataset_socioambiental_60.csv'

## 1.1. (EDA Rápida) Exploração Inicial do Dataset
Objetivo: entender estrutura, tipos, estatísticas e valores ausentes para guiar preparação.

In [ ]:
# =====================================================
# EDA Rápida (robusta) — evita erros quando não há colunas de certo tipo
# =====================================================
import numpy as np

print("\n=== Info do DataFrame ===")
df.info()

# Identificar listas de colunas por tipo
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("\nDetectadas:")
print("• Numéricas:", num_cols if len(num_cols) else "(nenhuma)")
print("• Categóricas:", cat_cols if len(cat_cols) else "(nenhuma)")

# Estatísticas de numéricas (se houver)
if len(num_cols):
    print("\n=== Estatísticas (numéricas) ===")
    display(df[num_cols].describe().T)
else:
    print("\n[AVISO] Não há colunas numéricas para descrever.")

# Estatísticas de categóricas (se houver)
if len(cat_cols):
    print("\n=== Estatísticas (categóricas) ===")
    # describe('all') em categóricas traz count, unique, top, freq
    display(df[cat_cols].describe().T)

    # (Opcional) Top categorias por coluna
    print("\n=== Principais categorias por coluna (top 5) ===")
    for c in cat_cols:
        print(f"\nColuna: {c}")
        display(df[c].value_counts(dropna=False).head(5).to_frame("freq"))
else:
    print("\n[AVISO] Não há colunas categóricas (object/category/bool) para descrever.")

# Valores ausentes (todas as colunas)
print("\n=== Valores ausentes por coluna ===")
na_counts = df.isna().sum().sort_values(ascending=False)
display(na_counts.to_frame("n_missing"))


## Verificação do Balanceamento da Variável-Alvo

**Objetivo.** Entender a distribuição da variável-alvo `risco_socioambiental` (0/1) para decidir como avaliar e treinar os modelos.

**Leitura da média.** Para variáveis binárias (0/1), a **média** é igual à **proporção de 1**.  
- Ex.: média = 0,90 → ~90% dos casos são classe 1.

**Critério prático (regra de bolso).**  
- Se a **classe minoritária** tiver **< 20%** das amostras (proporção < 0,20), trataremos o problema como **desbalanceado**.  
- Em cenários desbalanceados, **acurácia** pode enganar; devemos priorizar **precision/recall/F1** e analisar a **matriz de confusão**.

**Decisões quando há desbalanceamento.**
- Usar **divisão estratificada** (train/test) para preservar as proporções.  
- Relatar **precision, recall e F1** por classe (não só acurácia).  
- Considerar:  
  - `class_weight="balanced"` (SVM, Árvore),  
  - técnicas de reamostragem (undersampling/oversampling) — opcional,  
  - ajustar limiar de decisão (threshold) — opcional,  
  - validação cruzada estratificada.

A seguir, checamos **contagens, proporções e média** e geramos um **gráfico** para inspecionar a distribuição.


## 1.2. Checar balanceamento da variável-alvo (0/1) 

In [ ]:

# Objetivo:
#   - Definir o TARGET (nome da coluna-alvo) em um só lugar;
#   - Ver contagens e proporções das classes;
#   - Entender a média (em 0/1 = proporção de 1);
#   - Sinalizar desbalanceamento (regra de bolso);
#   - (Opcional) Fazer split estratificado e conferir proporções no treino/teste.
# Pré-requisitos:
#   - df já carregado (ex.: df = pd.read_csv(DATAPATH)).
# =====================================================



# 0) Definir alvo (boa prática) 
TARGET = "risco_socioambiental"   # nome da coluna que queremos prever (ajuste se necessário)
assert TARGET in df.columns, f"Coluna-alvo '{TARGET}' não encontrada em df.columns"  # checa se a coluna existe

# 1) Contagens e proporções no dataset completo 
counts = df[TARGET].value_counts().sort_index()              # conta quantos 0 e quantos 1; ordena por rótulo
props  = df[TARGET].value_counts(normalize=True).sort_index()# mesma contagem, mas em proporção (0 a 1), por classe

print("=== Distribuição no dataset completo ===")             # título amigável no console
# junta contagens e proporções em uma tabelinha para visualização
display(pd.DataFrame({"contagem": counts, "proporcao": props.round(4)}))

# 2) Média da variável-alvo (binária) 
mean_target = df[TARGET].mean()                              # em 0/1, a média = proporção da classe 1
print(f"\nMédia da variável-alvo (proporção da classe 1): {mean_target:.4f}")  # exibe com 4 casas decimais

# 3) Sinalizador simples de desbalanceamento 
minority_prop = props.min() if len(props) > 0 else np.nan    # pega a menor proporção (classe minoritária)
is_imbalanced = (minority_prop < 0.20)                       # regra de bolso: <20% é considerado desbalanceado
print(f"Classe minoritária — proporção: {minority_prop:.4f}")# mostra a proporção da classe menor
print("Desbalanceado? ", "SIM" if is_imbalanced else "NÃO")  # imprime um rótulo amigável

# 4) Visualização rápida 
plt.figure(figsize=(4,3))                                    # cria figura pequena (4x3 polegadas)
counts.plot(kind="bar")                                      # gráfico de barras com as contagens por classe
plt.title(f"Distribuição de classes — {TARGET}")             # título do gráfico
plt.xlabel("Classe")                                         # rótulo do eixo x
plt.ylabel("Contagem")                                       # rótulo do eixo y
plt.tight_layout()                                           # ajusta espaçamentos para não cortar textos
plt.show()                                                   # exibe o gráfico

# 5) (Opcional) Split estratificado 
X = df.drop(columns=[TARGET])                                 # X = todas as colunas exceto a variável-alvo
y = df[TARGET].copy()                                         # y = vetor/coluna com os rótulos (0/1)

# divide treino/teste mantendo proporções (stratify=y)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,                      # dados de entrada (X) e rótulos (y)
    test_size=0.30,            # 30% para teste (ajuste para 0.2/0.25 se preferir)
    random_state=42,           # semente para reprodutibilidade (mesmo split toda vez)
    stratify=y                 # preserva a proporção das classes em treino e teste
)

print("\n=== Proporções por classe (após split) ===")         # título amigável para as proporções pós-split

# calcula e mostra a proporção por classe no TREINO; to_frame/T transfornam em tabela de 1 linha
proporcao_treino = y_train.value_counts(normalize=True).sort_index().to_frame("proporcao_treino").T
display(proporcao_treino)                                    # exibe a tabela de proporções do treino

# idem para o TESTE
proporcao_teste = y_test.value_counts(normalize=True).sort_index().to_frame("proporcao_teste").T
display(proporcao_teste)                                     # exibe a tabela de proporções do teste

# 6) Orientação textual (para o relatório/console) 
if is_imbalanced:
    # recomendações quando há desbalanceamento
    print(
        "\n[Orientação] O conjunto está desbalanceado.\n"
        "- Na avaliação, priorize Precision, Recall, F1 e Matriz de Confusão (não só Acurácia).\n"
        "- Considere class_weight='balanced' (em SVM/Árvore) e/ou reamostragem (SMOTE/undersampling).\n"
        "- Use validação cruzada estratificada.\n"
        "- Reflita sobre o custo de FP vs FN no seu contexto (COP30: mais grave aprovar projeto fraco ou reprovar um bom?)."
    )
else:
    # recomendações quando o conjunto não está fortemente desbalanceado
    print(
        "\n[Orientação] O conjunto não parece fortemente desbalanceado.\n"
        "- Siga com avaliação padrão, mas ainda reporte métricas por classe e a matriz de confusão."
    )


# 2. Preparação dos dados

In [ ]:
# Objetivo:
# - Separar X(preditoras/features) e y (alvo) usando o TARGET já definido.
# - Fazer split treino/teste com estratificação )dados desbalanceados).
# - Criar preprocessadores simples:
#      - Para Árvore: imputação (sem padronização).
#      - Para SVM/K-NN: imputação + padronizaçao (StandardScaler).
# Observação: não há colunas cetegóricas; logo, não usamos OneHotEncoder


#1) Separa X e y
y = df[TARGET].copy()                 # y = coluna-alvo (0/1), já definida em bloco anterior
X = df.drop(columns=[TARGET])         # X = todas as colunas exceto o TARGET

# 2) Sanidade: Garantir que tudo em X é numérico
# Por que: preprocessadores abaixo asusmem somente colunas numéricas

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()  # lista de colunas numéricas detectadas
assert set(num_cols) == set(X.columns), (
    "Foram encontradas colunas não numéricas em X. "
    "Converta-as para numéricas ou remova antes de prosseguir."
)

print("Colunas numéricas:", num_cols)


# 3) Split treino/teste com estratificação
SEED = 42          # semente p/ reprodutibilidade (definida no bloco 1; repito aqui por clareza local)
TEST_SIZE = 0.30   # 30% para teste (ajuste se preferir 0.2/0.25)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y         # preserva proporção das classes (importante em desbalanceamento)
)

print(f"\nDimensões -> X_train: {X_train.shape} | X_test: {X_test.shape} | y_train: {y_train.shape} | y_test: {y_test.shape}")
print("Proporções treino:", y_train.value_counts(normalize=True).round(3).to_dict())
print("Proporções teste: ", y_test.value_counts(normalize=True).round(3).to_dict())


#4) Preprocessadores
# Ideia: encapsular a preparação em objetos reutilizáveis para acoplar aos modelos no Bloco 4.

# Para Árvore de Decisão:
# - Árvores não precisam de padronização (baseadas em cortes).
# - Mesmo sem nulos, mantemos SimpleImputer(median) por robustez (não altera se não houver NaN).
preprocess_tree = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))  # se não houver nulos, não muda nada
    # (sem scaler)
])

# Para SVM/k-NN:
# - SVM e k-NN são sensíveis à escala → aplicamos StandardScaler.
# - Mantemos o imputer por robustez.
preprocess_scaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("\nResumo dos preprocessadores criados:")
print("- preprocess_tree: SimpleImputer(median)  [sem scaler]")
print("- preprocess_scaled: SimpleImputer(median) + StandardScaler()")

## Interpretação da Preparação dos Dados

**Colunas numéricas detectadas**  
Foram identificadas 4 variáveis preditoras, todas numéricas:  
- `desmatamento_pct` — percentual de desmatamento associado ao projeto.  
- `poluicao_mgL` — concentração de poluição em mg/L.  
- `dist_area_protegida_km` — distância até a área protegida mais próxima (km).  
- `emissao_co2_ton` — emissões de CO₂ em toneladas.  

Não foram detectadas variáveis categóricas, o que simplifica o pré-processamento (não é necessário aplicar *one-hot encoding*).

---

**Divisão treino/teste (estratificada)**  
- Treino: 42 observações (70% do total).  
- Teste: 18 observações (30% do total).  
- A estratificação preservou a proporção de classes:  
  - Classe 1 (risco alto) ≈ 90%  
  - Classe 0 (risco baixo) ≈ 10%  

Isso confirma o desbalanceamento já observado anteriormente (classe majoritária = 1).

---

**Preprocessadores criados**  
- **Para Árvore de Decisão (`preprocess_tree`)**:  
  - Nenhuma transformação relevante é necessária, pois o algoritmo não depende de padronização de escala e não há valores ausentes.  
  - O pipeline foi mantido simples, apenas como estrutura para integrar ao modelo.

- **Para SVM/k-NN (`preprocess_scaled`)**:  
  - Aplicação de `StandardScaler` nas variáveis numéricas, já que esses algoritmos são sensíveis à escala dos dados.  
  - Como não há valores ausentes, nenhuma etapa de imputação é necessária neste caso.

---

**Conclusão da Preparação**  
- O dataset está pronto para a etapa de **modelagem**.  
- Como a variável-alvo está **fortemente desbalanceada**, os modelos que suportam `class_weight` serão configurados com `class_weight="balanced"`.  
- A avaliação deve priorizar métricas além da acurácia, como precisão, recall, F1 e matriz de confusão, para capturar melhor o desempenho em um cenário desbalanceado.


# 3. Modelagem 
Objetivo:
1) Criar pipelines para os 3 algoritmos escolhidos:
 - Decision Tree (Árvore de Decisão)
 - SVM (Support Vector Machine)
 - k-NN (K-Nearest Neighbors)
2) Lidar com desbalanceamento usando class_weight="balanced"
         (suportado em DecisionTree e SVM).
3) Associar cada modelo ao preprocessador adequado:
 - Árvore -> preprocess_tree  (não precisa de padronização)
 - SVM/k-NN -> preprocess_scaled (precisam de padronização)
4) Organizar todos os modelos em um dicionário para fácil iteração.

In [ ]:
# 1) Pipeline para Decision Tree 
# Árvore de decisão funciona com dados em diferentes escalas, então não aplicamos scaler.
# Adicionamos class_weight="balanced" para lidar com o desbalanceamento das classes.

pipe_tree = Pipeline(steps=[
    ("prep", preprocess_tree),                  # aplica o pré-processamento definido no Bloco 3
    ("clf", DecisionTreeClassifier(
        criterion="gini",                       # função de impureza ("gini" ou "entropy")
        max_depth=5,                            # profundidade máxima da árvore (evita overfitting)
        class_weight="balanced",                # compensa classes desbalanceadas
        random_state=SEED                       # semente para reprodutibilidade
    ))
])

# 2.) Pipeline para SVM 
# SVM é sensível à escala, então precisa do preprocess_scaled (com StandardScaler).
# Também suporta class_weight="balanced".

pipe_svm = Pipeline(steps=[
    ("prep", preprocess_scaled),                # inclui padronização
    ("clf", SVC(
        kernel="rbf",                           # kernel radial (não linear)
        probability=True,                       # permite calcular probabilidades (necessário para ROC)
        class_weight="balanced",                # compensa o desbalanceamento
        random_state=SEED
    ))
])

# 3.) Pipeline para k-NN 
# k-NN também é sensível à escala, então usamos preprocess_scaled.
# OBS: k-NN não tem parâmetro class_weight, então só usa o pré-processamento.

pipe_knn = Pipeline(steps=[
    ("prep", preprocess_scaled),
    ("clf", KNeighborsClassifier(
        n_neighbors=5                           # número de vizinhos a considerar (hiperparâmetro ajustável)
    ))
])

# 4.) Organizar modelos em um dicionário 
# Isso facilita o loop na avaliação (Bloco 5).
models = {
    "DecisionTree": pipe_tree,
    "SVM": pipe_svm,
    "kNN": pipe_knn
}

print("Modelos configurados e prontos para treino:")
for name in models:
    print("-", name)

# 4. Avaliação
1. Objetivo:
   - Treinar cada pipeline (modelo) com X_train/y_train;
   - Gerar previsões em X_test;
   - Calcular métricas: Acurácia, Precisão, Recall, F1;
   - Exibir Matriz de Confusão (com TN/FP/FN/TP);
   - Imprimir classification_report (por classe);
   - Plotar Curva ROC e AUC quando possível (binário);
   - Consolidar resultados em uma tabela comparativa.
2. Pré-requisitos:
   - models (dicionário com pipelines) do Bloco 4
   - X_train, X_test, y_train, y_test do Bloco 3
   - matplotlib importado como plt


In [ ]:
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    """
    Treina e avalia um pipeline de classificação.
    Retorna um dicionário com métricas agregadas para comparação.
    """
    print(f"\n================= {name} =================")     # Título do bloco de avaliação

    # 1) Treinar o pipeline no conjunto de treino
    model.fit(X_train, y_train)

    # 2) Prever no conjunto de teste
    y_pred = model.predict(X_test)

    # 3) Definir se é problema binário
    is_binary = (y_test.nunique() == 2)
    avg = "binary" if is_binary else "macro"

    # 4) Calcular métricas agregadas
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average=avg, zero_division=0)
    rec  = recall_score(y_test, y_pred, average=avg, zero_division=0)
    f1   = f1_score(y_test, y_pred, average=avg, zero_division=0)

    print(f"Acurácia:  {acc:.4f}")
    print(f"Precisão:  {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}\n")

    # 5) Matriz de Confusão
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.title(f"Matriz de Confusão — {name}")
    plt.show()

    if is_binary and cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        print("Contagens (binário):")
        print(f" TN (Verdadeiro Negativo): {tn}")
        print(f" FP (Falso Positivo):      {fp}")
        print(f" FN (Falso Negativo):      {fn}")
        print(f" TP (Verdadeiro Positivo): {tp}\n")

    # 6) Relatório por classe
    print("Relatório de Classificação (por classe):")
    print(classification_report(y_test, y_pred, zero_division=0))

    # 7) Curva ROC-AUC (se binário e suportado)
    if is_binary:
        if hasattr(model.named_steps["clf"], "predict_proba"):
            y_score = model.predict_proba(X_test)[:, 1]
        elif hasattr(model.named_steps["clf"], "decision_function"):
            y_score = model.decision_function(X_test)
        else:
            y_score = None

        if y_score is not None:
            fpr, tpr, _ = roc_curve(y_test, y_score)
            roc_auc = auc(fpr, tpr)
            plt.figure()
            plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
            plt.plot([0, 1], [0, 1], linestyle="--")
            plt.xlabel("False Positive Rate (1 - Especificidade)")
            plt.ylabel("True Positive Rate (Recall/Sensibilidade)")
            plt.title(f"Curva ROC — {name}")
            plt.legend(loc="lower right")
            plt.show()
        else:
            print("Curva ROC-AUC indisponível para este modelo (sem scores contínuos).")

    # 8) Retornar métricas
    return {
        "model": name,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1
    }
# 9) Rodar avaliação para todos os modelos
results = []
for name, mdl in models.items():
    metrics_dict = evaluate_model(
        name=name,
        model=mdl,
        X_train=X_train, y_train=y_train,
        X_test=X_test,   y_test=y_test
    )
    results.append(metrics_dict)

# 10) Tabela de comparação
results_df = pd.DataFrame(results).sort_values("f1", ascending=False).reset_index(drop=True)
print("\nComparação de Modelos (ordenado por F1):")
display(results_df)


## Interpretação dos Resultados da Avaliação

### Desempenho dos Modelos
Todos os três algoritmos testados — **Árvore de Decisão, SVM e k-NN** — atingiram **100% em todas as métricas** (Acurácia, Precisão, Recall e F1) no conjunto de teste.  
A matriz de confusão mostrou que, entre as 18 amostras de teste, **todas foram classificadas corretamente** (16 positivos e 2 negativos).

### Como interpretar resultados perfeitos?
- Em cenários reais, métricas perfeitas são **raras** e devem ser analisadas com cautela.  
- Possíveis explicações:
  1. **Separabilidade muito alta** — uma ou duas variáveis já distinguem perfeitamente as classes.
  2. **Amostra de teste pequena e desbalanceada** — apenas 2 exemplos da classe 0 foram avaliados.
  3. **Sorte no split** — a divisão treino/teste, mesmo estratificada, pode gerar partições fáceis.

### Necessidade de Validação Adicional
Embora não tenhamos encontrado problemas de qualidade dos dados na exploração inicial, a magnitude dos resultados justifica **checagens adicionais** em um projeto profissional:
- **Validação cruzada estratificada** com múltiplas repetições, para verificar se o desempenho se sustenta em diferentes partições.  
- **Análise de importância das variáveis** (ex.: importâncias na Árvore ou coeficientes de um SVM linear), para identificar quais atributos explicam a separação.  
- **Curvas de aprendizagem** (opcional), para observar se o modelo mantém desempenho alto mesmo com menos dados de treino.

### Conclusão parcial
- Os modelos indicam que o dataset possui forte capacidade preditiva: os atributos ambientais analisados distinguem bem entre projetos de **alto** e **baixo risco socioambiental**.  
- Entretanto, em função do **tamanho reduzido da amostra de teste** e do **desbalanceamento de classes**, é essencial avançar para a etapa de **validação cruzada** (Bloco 6) antes de considerar esses resultados como conclusivos.  
- Este cuidado aumenta a **credibilidade científica** e garante que as conclusões apoiem decisões robustas no contexto da **COP30**.


# 5. Validaçao cruzada estatificada
Objetivo:
- Avaliar estabilidade dos modelos além do split único.
- Usar RepeatedStratifiedKFold (K-fold repetido).
- Calcular média ± desvio para várias métricas.

In [ ]:
# 1) Configuração do avaliador CV 
# 5 dobras, repetido 5 vezes (total = 25 avaliações).
# Estratificação garante mesma proporção de classes em cada dobra.

from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
cv = RepeatedStratifiedKFold(
    n_splits=5,        # número de dobras
    n_repeats=5,       # número de repetições
    random_state=SEED  # para reprodutibilidade
)

# 2) Métricas que queremos avaliar 
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

# 3) Função auxiliar para rodar CV 
def cv_report(name, model, X, y, cv, scoring):
    """
    Executa validação cruzada com várias métricas.
    Retorna resultados e imprime médias ± desvios.
    """
    print(f"\n=== Validação Cruzada — {name} ===")
    
    # cross_validate executa treino/teste em cada dobra
    out = cross_validate(
        estimator=model,      # pipeline do modelo (já com preprocessador)
        X=X, y=y,             # dados completos (X e y)
        cv=cv,                # estratégia de validação
        scoring=scoring,      # métricas
        n_jobs=None,          # usa 1 CPU (pode ser -1 para usar todas)
        return_train_score=False
    )
    
    # imprimir resultados resumidos
    for m in scoring.keys():
        mean = out[f"test_{m}"].mean()
        std  = out[f"test_{m}"].std()
        print(f"{m:>9}: {mean:.3f} ± {std:.3f}")
    
    return out

# 4) Rodar CV para cada modelo 
cv_tree = cv_report("DecisionTree", models["DecisionTree"], X, y, cv, scoring)
cv_svm  = cv_report("SVM",          models["SVM"],          X, y, cv, scoring)
cv_knn  = cv_report("kNN",          models["kNN"],          X, y, cv, scoring)

# 5) Consolidar resultados em DataFrame 
import pandas as pd

# cria um resumo em tabela (médias das métricas)
summary = []
for name, cv_out in zip(["DecisionTree", "SVM", "kNN"], [cv_tree, cv_svm, cv_knn]):
    row = {"model": name}
    for m in scoring.keys():
        row[m] = cv_out[f"test_{m}"].mean()
    summary.append(row)

cv_results_df = pd.DataFrame(summary).sort_values("f1", ascending=False).reset_index(drop=True)

print("\nResumo comparativo (médias de CV):")
display(cv_results_df)

## Interpretação dos Resultados — Validação Cruzada (Bloco 6)

### Resumo geral
A validação cruzada estratificada (5 dobras × 5 repetições) confirmou o excelente desempenho observado nos testes iniciais:

- **k-NN** — F1 = **0,986 ± 0,022**, ROC-AUC ≈ **0,999**
- **SVM (RBF)** — F1 = **0,982 ± 0,024**, ROC-AUC = **1,000**
- **Decision Tree** — F1 = **0,947 ± 0,040**, ROC-AUC ≈ **0,744**

### Principais insights
- **Consistência:** k-NN e SVM apresentaram métricas muito altas com baixa variabilidade, reforçando a robustez dos modelos.  
- **Árvore de Decisão:** apesar de bons valores de precisão, recall e F1, o ROC-AUC foi inferior (~0,74). Isso sugere que o modelo acerta bem com limiar fixo (0,5), mas tem dificuldade em **ordenar corretamente os casos** por probabilidade.  
- **Desbalanceamento:** a estratégia de estratificação manteve a proporção de classes em todas as dobras, garantindo avaliação justa mesmo com ≈90% da classe positiva.

### Conclusão parcial
- **Modelo com melhor desempenho:** **k-NN** (F1 mais alto e AUC próximo de 1).  
- **Modelo alternativo:** **SVM (RBF)**, praticamente empatado em desempenho, com AUC perfeito.  
- **Modelo interpretável:** **Árvore de Decisão**, embora inferior em separação, segue útil para explicar regras de decisão de forma simples.

### Implicações para o projeto
- O dataset apresenta atributos com **forte poder discriminatório**, especialmente relevantes para diferenciar projetos de alto e baixo risco socioambiental.  
- Para uso em **apoio à decisão no contexto COP30**, recomenda-se priorizar modelos mais robustos (k-NN ou SVM) e manter a Árvore como recurso explicativo em comunicações executivas.  
- Antes da etapa de deploy, será importante avançar para:
  - **Ajuste de hiperparâmetros** (refinamento fino de k-NN e SVM);
  - **Calibração de probabilidades** (especialmente para a Árvore);
  - **Análise de custos de erro** (impacto relativo de FP vs. FN no processo de avaliação de projetos).

---


# 6. Ajustes Avançados
Objetivo:
1) Refinar hiperparâmetros de k-NN e SVM.
2) Calibrar probabilidades da Árvore de Decisão.
3) Analisar custos de erro ajustando o limiar de decisão.

In [ ]:

from sklearn.calibration import CalibratedClassifierCV

# 1) Funções auxiliares para escolher n_splits seguro 
def safe_n_splits(y, max_splits=5, min_splits=2):
    """
    Retorna um número de dobras que não excede a contagem da classe minoritária.
    Garante ao menos 'min_splits' (típico=2 ou 3).
    """
    min_cls = int(y.value_counts().min())
    return max(min_splits, min(max_splits, min_cls))

# folds seguros para os diferentes contextos:
cv_splits_global = safe_n_splits(y, max_splits=5, min_splits=3)       # para GridSearch em X,y
cv_splits_train  = safe_n_splits(y_train, max_splits=5, min_splits=3) # para calibração em X_train,y_train

print(f"CV (global, GridSearch): {cv_splits_global} folds")
print(f"CV (train, calibração):  {cv_splits_train} folds")

# Também criamos objetos StratifiedKFold explícitos (opcional, só para clareza)
cv_global = StratifiedKFold(n_splits=cv_splits_global, shuffle=True, random_state=SEED)
cv_train  = StratifiedKFold(n_splits=cv_splits_train,  shuffle=True, random_state=SEED)

# 2) Ajuste de Hiperparâmetros 
# k-NN: vizinhos e ponderação
param_knn = {
    "clf__n_neighbors": [3, 5, 7, 11],
    "clf__weights": ["uniform", "distance"]
}

grid_knn = GridSearchCV(
    estimator=models["kNN"],        # pipeline com preprocess_scaled + KNeighborsClassifier
    param_grid=param_knn,
    scoring="f1",                   # métrica mais sensível no desbalanceamento
    cv=cv_global,                   # usa número de dobras seguro para X,y
    n_jobs=-1
)
grid_knn.fit(X, y)
print("\n[GridSearch] Melhor k-NN:", grid_knn.best_params_)
print("F1 médio (CV):", round(grid_knn.best_score_, 4))

# SVM (RBF): C e gamma
param_svm = {
    "clf__C": [0.5, 1, 2],
    "clf__gamma": ["scale", 0.1, 0.01]
}

grid_svm = GridSearchCV(
    estimator=models["SVM"],        # pipeline com preprocess_scaled + SVC(probability=True)
    param_grid=param_svm,
    scoring="f1",
    cv=cv_global,
    n_jobs=-1
)
grid_svm.fit(X, y)
print("\n[GridSearch] Melhor SVM:", grid_svm.best_params_)
print("F1 médio (CV):", round(grid_svm.best_score_, 4))

# ---------- 7.2) Calibração de Probabilidades (Árvore) ----------
# Usamos CalibratedClassifierCV no classificador da árvore,
# com cv baseado em y_train (pois vamos ajustar em X_train,y_train).
tree_base = models["DecisionTree"].named_steps["clf"]     # pega o estimador da árvore do pipeline
tree_calibrated = CalibratedClassifierCV(
    base_estimator=tree_base,
    method="isotonic",   # isotonic é mais flexível; 'sigmoid' (Platt) é alternativa mais simples
    cv=cv_train          # usa folds compatíveis com a classe minoritária no TREINO
)

# Reencaixamos num pipeline com o preprocessamento da árvore
pipe_tree_calibrated = Pipeline(steps=[
    ("prep", preprocess_tree),
    ("clf", tree_calibrated)
])
pipe_tree_calibrated.fit(X_train, y_train)
print("\nÁrvore calibrada treinada com calibração isotônica.")

# (Opcional) Avaliar rapidamente no teste para ver se AUC melhora:
from sklearn.metrics import roc_auc_score, roc_curve, auc
y_score_tree = pipe_tree_calibrated.predict_proba(X_test)[:, 1]
auc_tree_cal = roc_auc_score(y_test, y_score_tree)
print("ROC-AUC (Árvore calibrada, teste):", round(auc_tree_cal, 3))

# 3) Análise de custos de erro (FP vs FN) via limiar 
# Escolhe o melhor entre k-NN e SVM pelo F1 médio do GridSearch
best_model = grid_knn.best_estimator_ if grid_knn.best_score_ >= grid_svm.best_score_ else grid_svm.best_estimator_
print("\nModelo selecionado para análise de custo:", "k-NN" if best_model is grid_knn.best_estimator_ else "SVM")

# Probabilidades no teste para classe positiva
y_proba = best_model.predict_proba(X_test)[:, 1]

# Definimos uma grade de limiares e um custo exemplo: FP=2, FN=5 (ajuste conforme sua política)
thresholds = np.linspace(0.1, 0.9, 9)
print("\nAnálise de custos (limiar → FP, FN, Custo=2*FP + 5*FN):")
for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
    custo = 2*fp + 5*fn   # <<< ajuste aqui os pesos conforme COP30
    print(f"  τ={t:.1f} → FP={fp:2d}, FN={fn:2d}, Custo={custo:2d}")
 

### 1) Ajuste de hiperparâmetros (Grid Search)
- **k-NN (pré-processado com StandardScaler)**  
  - Melhor configuração: `n_neighbors = 3`, `weights = "uniform"`.  
  - Desempenho (F1 médio em CV): **0,9895**.

- **SVM (RBF, com StandardScaler)**  
  - Melhor configuração: `C = 2`, `gamma = "scale"`.  
  - Desempenho (F1 médio em CV): **0,9905**.

**Leitura.** Ambos os modelos se mantiveram em **nível de excelência** após o refino, com leve vantagem do **SVM (RBF)** em F1 médio. A diferença é pequena e ambas as opções são tecnicamente válidas para produção.

---

### 2) Calibração de probabilidades (Decision Tree)
- Método: **Isotonic Regression** via `CalibratedClassifierCV`.  
- **ROC-AUC no teste:** **1,00** após calibração.

**Leitura.** A Árvore já performava bem no limiar fixo (precision/recall/F1 altos), mas tinha **AUC inferior** por problemas de ordenação/calibração de probabilidade. A calibração **corrigiu** esse ponto, elevando o AUC para 1,00 no teste. Isso torna a Árvore mais útil quando precisamos:
- comparar casos por “risco” (ranking/triagem),  
- ou ajustar o **limiar de decisão** conforme o custo FP/FN.

> **Observação técnica:** o scikit-learn sinalizou:  
> *FutureWarning: `base_estimator` foi renomeado para `estimator` na versão 1.2 e será removido na 1.4.*  
> Em versões mais novas, use `CalibratedClassifierCV(estimator=..., ...)`.

---

### 3) Análise de custos de erro (FP vs FN) — ajuste de limiar
- Modelo escolhido para análise: **SVM (RBF)** (melhor F1 médio no Grid).  
- Custos exemplificativos: **FP = 2**, **FN = 5** (ajustável conforme política COP30).  
- Resultado (teste): para **τ ∈ [0,2, 0,9]**, o custo permaneceu **zero** (FP=0, FN=0).  
  - Em **τ = 0,1**, houve **2 FP** (custo = 4).  
  - Todos os demais limiares testados mantiveram **FP = 0** e **FN = 0**.

**Leitura.** Na partição de teste, o SVM separou perfeitamente a classe positiva para limiares entre 0,2 e 0,9 — isto é, **robusto a variações de limiar**. Em **produção**, ainda assim recomendamos:
- **especificar custos** (qual é mais grave: aprovar projeto fraco — FP — ou reprovar projeto bom — FN);  
- **definir limiar operacional** de acordo com essa política;  
- **monitorar drift** (se a distribuição mudar, o limiar ótimo pode mudar).

---

## Conclusão técnica e recomendação prática

- **Modelo recomendado (baseline de produção):** **SVM (RBF)** com `C = 2` e `gamma = "scale"`, pelo melhor F1 médio e AUC consistente.  
- **Alternative de alto desempenho:** **k-NN (k=3)**, praticamente empatado e simples.  
- **Modelo explicável:** **Árvore de Decisão**, agora **calibrada** e com AUC excelente — ótima para **explicar regras** e **comunicar critérios** a stakeholders.

**Para COP30**, onde decisões sobre projetos têm custos assimétricos, mantenha:
1) **Threshold tuning** alinhado aos custos FP/FN definidos pela governança do programa;  
2) **Relatórios de interpretabilidade** (importância/SHAP/permutation) com 2–3 insights acionáveis (ex.: “emissão de CO₂ e poluição são os vetores mais discriminativos”);  
3) **Plano de monitoramento** pós-deploy (recalibrar a cada trimestre ou quando houver novos lotes de projetos).

---

## Próximos passos operacionais (sugeridos)

1) **Reajuste final em todos os dados** com os hiperparâmetros escolhidos.  
2) **Persistência do pipeline** (`joblib.dump`) para uso em produção.  
3) **Script de inferência** de exemplo (carrega o modelo e prevê a partir de um CSV).  
4) **Documento de governança do limiar** (quem ajusta, quando e com base em quê).  
5) **Dashboard de monitoramento** (distribuições por feature, taxa de positividade, FP/FN, F1, ROC-AUC ao longo do tempo).

## 7. Reajuste final + persistência do pipeline

Objetivo:
- Reajustar o melhor modelo (segundo o Grid) em TODOS os dados (X, y);
- Persistir o pipeline completo (pré-processamento + classificador);
- Salvar metadados de versão e hiperparâmetros para rastreabilidade.

Pré-requisitos:
- grid_svm e grid_knn já executados 
- X, y definidos (Bloco 3).

In [ ]:

import json, joblib, time, sys, platform
from pathlib import Path

# 1) Escolher o melhor entre SVM e k-NN pelo F1 médio do Grid
best_grid = grid_svm if grid_svm.best_score_ >= grid_knn.best_score_ else grid_knn
best_name = "SVM" if best_grid is grid_svm else "kNN"
best_model = best_grid.best_estimator_    # pipeline completo (prep + clf)
best_params = best_grid.best_params_

print(f"Melhor modelo (Grid): {best_name} | params = {best_params} | F1-CV = {best_grid.best_score_:.4f}")

# 2) Reajustar em TODOS os dados disponíveis (X, y) para produção
# Por quê: queremos que o modelo final conheça o máximo de informação possível.
best_model.fit(X, y)

# 3) Persistir o pipeline
out_dir = Path("models")
out_dir.mkdir(parents=True, exist_ok=True)

model_path = out_dir / f"cop30_{best_name.lower()}_pipeline.joblib"
joblib.dump(best_model, model_path)

print(f"Pipeline salvo em: {model_path.resolve()}")

# 4) Salvar metadados (versões, hiperparâmetros, tempo, colunas)
metadata = {
    "model_name": best_name,
    "best_params": best_params,
    "f1_cv_mean": round(best_grid.best_score_, 6),
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "python": sys.version.split()[0],
    "os": f"{platform.system()} {platform.release()}",
    "features": list(X.columns),
    "target": TARGET
}

meta_path = out_dir / f"cop30_{best_name.lower()}_metadata.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Metadados salvos em: {meta_path.resolve()}")


In [ ]:
#Verificar rapidademente os artefatos salvos

model_path = Path("models/cop30_svm_pipeline.joblib")
meta_path  = Path("models/cop30_svm_metadata.json")

pipe = joblib.load(model_path)
with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

print("OK: modelo carregado.")
print("Hiperparâmetros:", meta["best_params"])
print("F1 médio (CV):", meta["f1_cv_mean"])
print("Features:", meta["features"])


## 8. Criar CSV de novos projetos

- Objetivo: gerar um arquivo de exemplo com as mesmas colunas usadas no treino, mas sem a variável-alvo (risco_socioambiental).
- Justificativa: em produção, o modelo recebe apenas os "features" de entrada e precisa gerar as probabilidades/predições.

In [ ]:
# Criamos um pequeno DataFrame com 5 projetos fictícios.
# - As colunas devem ter exatamente os mesmos nomes do treino.
# - Os valores são apenas de teste (valores plausíveis, mas inventados).
novos = pd.DataFrame({
    "desmatamento_pct": [15.0, 60.0, 30.0, 45.0, 22.5],
    "poluicao_mgL": [40.2, 80.1, 20.0, 55.0, 33.3],
    "dist_area_protegida_km": [5.0, 12.0, 3.5, 10.0, 7.0],
    "emissao_co2_ton": [500, 1800, 700, 1200, 650]
})

# Exportamos esse DataFrame para CSV.
# Esse arquivo será a entrada do script infer.py.
novos.to_csv("novos_projetos.csv", index=False)
print("Arquivo 'novos_projetos.csv' criado com sucesso.")


### Criação de arquivo de novos projetos (novos_projetos.csv)
Para validar o pipeline em condições de produção, foi criado um CSV contendo apenas as variáveis de entrada (features) do modelo: **desmatamento_pct**, **poluicao_mgL**, **dist_area_protegida_km** e **emissao_co2_ton**.  
Esse arquivo simula projetos que ainda não possuem classificação (`risco_socioambiental`) e precisam ser avaliados pelo modelo.



## 9. Rodar script de inferência
- Objetivo: aplicar o pipeline salvo (cop30_svm_pipeline.joblib)aos novos projetos e gerar predições.
- Justificativa: simula como o modelo será usado em produção, recebendo um CSV,produzindo probabilidades e rótulos.

### Criação do script de inferência (`infer.py`)
Para possibilitar o uso do modelo fora do notebook, foi criado um script em Python chamado **`infer.py`**.  
Esse script recebe três argumentos principais:
- `--model`: caminho do pipeline salvo (`.joblib`);
- `--input`: CSV contendo os novos projetos a serem avaliados;
- `--output`: nome do arquivo de saída com as predições.

Essa separação é uma prática recomendada de **engenharia de MLOps**, pois permite que o modelo seja usado em diferentes contextos (notebooks, linha de comando, sistemas externos).

---

### Execução do script de inferência
O script `infer.py` pode ser executado a partir do terminal com o seguinte comando:

```bash
python scripts/infer.py \
  --model models/cop30_svm_pipeline.joblib \
  --input data/novos_projetos.csv \
  --output data/predicoes_novos_projetos.csv \
  --threshold 0.5


### Validação do fluxo
Após rodar o script, as predições foram inspecionadas no notebook para confirmar:
1. O modelo foi carregado corretamente;
2. As colunas de entrada foram reconhecidas;
3. O arquivo de saída foi gerado com probabilidades e rótulos esperados.

Esse processo comprova a viabilidade do uso do pipeline em **produção**, garantindo reprodutibilidade e clareza do fluxo.

### Do treino à produção
O modelo foi treinado e validado no arquivo `dataset_socioambiental_60.csv` (contendo 60 registros rotulados).  
Após o treinamento, o pipeline final foi salvo e aplicado em um novo conjunto (`predicoes_novos_projetos.csv`) para simular o uso em produção.




## 10. Visualizar predições
- Objetivo: inspecionar o arquivo de saída gerado pelo infer.py para verificar se as predições fazem sentido.
- Justificativa: garante que o fluxo completo (treino → salvamento →inferência em novos dados) está funcionando corretamente.




In [27]:
pred = pd.read_csv("data/predicoes_novos_projetos.csv")
display(pred.head())


,desmatamento_pct,poluicao_mgL,dist_area_protegida_km,emissao_co2_ton,prob_pos,pred_padrao_0_1,pred_thresh_0_1
0,15.0,40.2,5.0,500,0.952841,1,1
1,60.0,80.1,12.0,1800,0.988330,1,1
2,30.0,20.0,3.5,700,0.971068,1,1
3,45.0,55.0,10.0,1200,0.996344,1,1
4,22.5,33.3,7.0,650,0.874122,1,1


### Interpretação da tabela de predições
Cada linha é um novo projeto avaliado. As quatro primeiras colunas são as **features de entrada**.  
**prob_pos** é a probabilidade prevista de ser classe 1 (alto risco).  
**pred_padrao_0_1** é a decisão binária usando limiar fixo 0,5.  
**pred_thresh_0_1** é a decisão usando o **limiar da governança** (do `governanca.json`).

No nosso lote, todas as probabilidades ficaram altas (≥ 0,87), portanto todos os projetos foram classificados como **alto risco** nos dois critérios (padrão e governança).


# 11. Conclusão à partir das predições

In [29]:

import os, json, glob

# 1) Achar o arquivo de predições (pega o mais recente que começa com "predicoes")
cands = sorted(glob.glob(os.path.join("data", "predicoes*.csv")), key=os.path.getmtime)
assert cands, "Nenhum arquivo que comece com 'predicoes' encontrado em data/."
PRED_PATH = cands[-1]

# 2) Ler governança (threshold)
GOV_PATH = "governanca.json"
thresh = 0.5
if os.path.exists(GOV_PATH):
    with open(GOV_PATH, "r", encoding="utf-8") as f:
        govern = json.load(f)
    thresh = float(govern.get("threshold", 0.5))

# 3) Ler predições
pred = pd.read_csv(PRED_PATH)
assert "prob_pos" in pred.columns, f"O arquivo {PRED_PATH} não tem a coluna 'prob_pos'."
N = len(pred)

# 4) Métricas operacionais simples (sem rótulo verdadeiro)
n_pos = int((pred["prob_pos"] >= thresh).sum())
rate_pos = n_pos / N
med_prob = float(pred["prob_pos"].median())
p90_prob = float(pred["prob_pos"].quantile(0.90))

# 5) Top 5 projetos com maior risco (salva CSV)
top5 = pred.copy().nlargest(5, "prob_pos")
top5_path = os.path.join("data", "top5_alto_risco.csv")
top5.to_csv(top5_path, index=False)

# 6) Texto pronto para README (cole no README em "Resultados")
conclusao = f"""
**Resultados (predições operacionais)**  
- Arquivo avaliado: `{os.path.basename(PRED_PATH)}`  
- Limiar oficial (governança): **{thresh:.2f}**  
- Projetos avaliados (N): **{N}**  
- Projetos sinalizados como alto risco (≥ τ): **{n_pos}** ({rate_pos:.1%})  
- Mediana das probabilidades (prob_pos): **{med_prob:.3f}**  
- P90 das probabilidades: **{p90_prob:.3f}**  
- Exportado `top5_alto_risco.csv` com os 5 projetos mais críticos para revisão humana.
"""

print(conclusao)



**Resultados (predições operacionais)**  
- Arquivo avaliado: `predicoes_novos_projetos.csv`  
- Limiar oficial (governança): **0.50**  
- Projetos avaliados (N): **5**  
- Projetos sinalizados como alto risco (≥ τ): **5** (100.0%)  
- Mediana das probabilidades (prob_pos): **0.971**  
- P90 das probabilidades: **0.993**  
- Exportado `top5_alto_risco.csv` com os 5 projetos mais críticos para revisão humana.



### Interpretação da tabela de predições
Cada linha é um novo projeto avaliado. As quatro primeiras colunas são as **features de entrada**.  
**prob_pos** é a probabilidade prevista de ser classe 1 (alto risco).  
**pred_padrao_0_1** é a decisão binária usando limiar fixo 0,5.  
**pred_thresh_0_1** é a decisão usando o **limiar da governança** (do `governanca.json`).

No nosso lote, todas as probabilidades ficaram altas (≥ 0,87), portanto todos os projetos foram classificados como **alto risco** nos dois critérios (padrão e governança).
